# Benford Threshold Sweep Notebook

This notebook checks whether the weak Benford result is just a threshold problem.
It tries several grouping choices and several MAD cutoffs, then compares precision, recall, F1, and lift over the fraud base rate.


In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from pathlib import Path


In [2]:
from pathlib import Path

# This makes the notebook work whether you open it from the repo root
# or from inside the notebooks folder.
if (Path.cwd() / "data").exists():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").exists():
    REPO_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not find the repo root")

DATA_DIR = REPO_ROOT / "data" / "training" / "vynfi"
OUT_DIR = REPO_ROOT / "data" / "generated"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Data folder:", DATA_DIR)


Repo root: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint
Data folder: C:\Users\svgow\OneDrive\Documents\Shaachi\FinLint\FinLint\data\training\vynfi


In [3]:
# I try different grouping columns and minimum sizes to see where Benford works better.
configurations = [
    ("gl_account", 500),
    ("gl_account", 200),
    ("created_by", 300),
    ("cost_center", 300),
    ("profit_center", 500),
]
# Include the common MAD limits 0.006, 0.012, and 0.015 with nearby values for comparison.
thresholds = [0.004, 0.006, 0.008, 0.010, 0.012, 0.015, 0.020, 0.030]

# I join all three parquet parts before making the train and test split.
main_data = pd.concat(
    [
        pd.read_parquet(DATA_DIR / "train-00000-of-00003.parquet"),
        pd.read_parquet(DATA_DIR / "train-00001-of-00003.parquet"),
        pd.read_parquet(DATA_DIR / "train-00002-of-00003.parquet"),
    ],
    ignore_index=True,
)

empty_columns = [
    "auxiliary_account_number",
    "auxiliary_account_label",
    "lettrage",
    "lettrage_date",
    "tax_code",
]
leakage_columns = ["fraud_type", "anomaly_type", "is_anomaly"]
work_df = main_data.drop(columns=empty_columns, errors="ignore")

# The split is grouped by document_id to avoid putting one journal entry on both sides.
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_index, test_index = next(
    splitter.split(work_df, y=work_df["is_fraud"], groups=work_df["document_id"])
)

train_df = work_df.iloc[train_index].reset_index(drop=True).drop(columns=leakage_columns, errors="ignore")
test_df = work_df.iloc[test_index].reset_index(drop=True).drop(columns=leakage_columns, errors="ignore")
y_test = test_df["is_fraud"].astype(int)
base_rate = y_test.mean()
print("Base rate:", round(base_rate, 4))


Base rate: 0.0619


In [4]:
expected_digits = np.arange(1, 10)
expected_shares = np.log10(1 + 1 / expected_digits)

# These helper functions calculate the first digits and their distance from Benford.
def line_amounts(df):
    return (df["debit_amount"].fillna(0) + df["credit_amount"].fillna(0)).abs()


def first_digits(amount_series):
    usable = amount_series[amount_series >= 0.01]
    scaled = usable / np.power(10.0, np.floor(np.log10(usable)))
    return scaled.astype(int).clip(1, 9)


def digit_shares(digit_series):
    counts = digit_series.value_counts().reindex(expected_digits, fill_value=0).to_numpy(dtype=float)
    total = counts.sum()
    if total == 0:
        return np.zeros(9)
    return counts / total


def mad_score(observed):
    return float(np.mean(np.abs(observed - expected_shares)))


In [5]:
# I store every combination as one row so it is easy to sort and compare later.
rows = []

for group_column, min_rows in configurations:
    # Group scores are learned from training data and applied to the test rows.
    train_digits = first_digits(line_amounts(train_df))
    grouped = pd.DataFrame(
        {
            "group_value": train_df.loc[train_digits.index, group_column].to_numpy(),
            "digit": train_digits.to_numpy(),
        }
    )

    score_rows = []
    for group_value, chunk in grouped.groupby("group_value"):
        observed = digit_shares(chunk["digit"])
        score_rows.append(
            {
                "group_value": group_value,
                "rows": len(chunk),
                "mad": mad_score(observed),
                "testable": len(chunk) >= min_rows,
            }
        )

    group_scores = pd.DataFrame(score_rows)

    # Lower cutoffs flag more groups, while higher cutoffs are more strict.
    for threshold in thresholds:
        failing_groups = set(
            group_scores.loc[group_scores["testable"] & (group_scores["mad"] > threshold), "group_value"]
        )
        pred = test_df[group_column].isin(failing_groups).astype(int)
        flagged = int(pred.sum())

        # This avoids undefined metric values when a setting flags nothing.
        if flagged == 0:
            precision = 0.0
            recall = 0.0
            f1 = 0.0
            lift = 0.0
        else:
            precision = precision_score(y_test, pred, zero_division=0)
            recall = recall_score(y_test, pred, zero_division=0)
            f1 = f1_score(y_test, pred, zero_division=0)
            lift = precision / base_rate

        rows.append(
            {
                "group_column": group_column,
                "min_rows": min_rows,
                "threshold": threshold,
                "failing_groups": len(failing_groups),
                "rows_flagged": flagged,
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "f1": round(f1, 4),
                "lift_over_base_rate": round(lift, 3),
            }
        )

# Lift tells me whether the flagged rows contain more fraud than the normal base rate.
sweep_table = pd.DataFrame(rows)
sweep_table.to_csv(OUT_DIR / "benford_threshold_sweep.csv", index=False)
sweep_table.sort_values("lift_over_base_rate", ascending=False).head(15)


,group_column,min_rows,threshold,failing_groups,rows_flagged,precision,recall,f1,lift_over_base_rate
6,gl_account,500,0.020,10,1570,0.0650,0.0126,0.0210,1.049
14,gl_account,200,0.020,12,1771,0.0638,0.0139,0.0228,1.030
32,profit_center,500,0.004,86,92520,0.0637,0.7247,0.1170,1.028
12,gl_account,200,0.012,153,31264,0.0626,0.2407,0.0993,1.010
11,gl_account,200,0.010,250,55739,0.0621,0.4261,0.1084,1.003
4,gl_account,500,0.012,131,28860,0.0621,0.2204,0.0968,1.002
8,gl_account,200,0.004,497,130206,0.0621,0.9942,0.1168,1.002
0,gl_account,500,0.004,467,126785,0.0618,0.9639,0.1161,0.997
3,gl_account,500,0.010,226,53028,0.0617,0.4026,0.1070,0.996
9,gl_account,200,0.006,454,115598,0.0612,0.8703,0.1143,0.988
